# Sistema RSA para firmas digitales

En este notebook, mostraremos el proceso para generar las llaves pública y privada, así como el proceso para generar una firma digital y su verificación correspondiente usando el sistema RSA. Empezaremos por cargar las funciones que necesitamos:

In [8]:
import sys
import importlib.util

if 'google.colab' in sys.modules:
    runtime = "Google Colab"
    if importlib.util.find_spec("cryptocalc") is None:
        print("   Installing MA2006B from GitHub\n")
        !pip install git+https://github.com/Krul-dev/MA2006B.git
else:
    runtime = "Local environment"

import cryptocalc

print(f"\n========= Notebook execution context =========")
print(f"              Runtime: {runtime}")
print(f"       Python version: {sys.version.split()[0]}")
print(f"   CryptoCalc version: {cryptocalc.__version__}\n")

from cryptocalc import (
    rsa_key_generation,
    rsa_encryption,
    rsa_decryption,
    sha256_of_sentence,
)


========= Notebook execution context =========
              Runtime: Local environment
       Python version: 3.14.3
   CryptoCalc version: 0.1.0



## Protocolo RSA para la generación de llaves

Una vez que ya hemos cargado las librerías necesarias, empezamos por generar nuestras llaves *pública* y *privada.*

In [9]:
(public_key, private_key) = rsa_key_generation(1000)
d = private_key
e = public_key[0]
n = public_key[1]

print(f"Llave privada (d): {d}\n")
print(f"Llave pública (e): {e}\n")
print(f"Módulo para las llaves (n): {n}\n")

Llave privada (d): 12044315063987523507775498560815764295198285348366368249303302711575239810437413977234038190787124425127471664495632998027404680170208340647867144343361239014818042899360732762536488037423351575550083784768880571819486747568972457694827868929984158215658136585699131746533036123678039120703758329391467133823983234926495175680399503416470629856192092576663286157393278427762152250252895454618818294728634676521439569253637558743472167092722131798911123178852505816350552737841844783642002548162011477618471355909754035052789957913578941716990693898980367335524997640263344752254703296958515196353119473

Llave pública (e): 65537

Módulo para las llaves (n): 53515137379562734110446294859673406414536273008534689895226477953119084166551647445829638027770560911835871896681376257065899696563725018375543663649550204834856276440366396139549411288719606251310226508637161087141267998334084607453961630241652323863022867621489084560849870402541535583292353195479918471485017908983402471

Recordemos que el valor $d$ de la llave privada se debe de mantener en secreto. En cambio los valores de $e$ y $n$ corresponden a la llave pública y son conocidos por todos los agentes involucrados, incluída Eva.

## Protocolo RSA para la firma de mensajes

Para ilustrar el algoritmo de generación de firmas digitales, supongamos que Alicia desea firmar el siguiente mensaje llano $m$:

In [10]:
m = "Hello World!"
h = sha256_of_sentence(m)

print(f"Mensaje llano (m): {m}\n")
print(f"Hash del mensaje llano (h): {h}\n")

Mensaje llano (m): Hello World!

Hash del mensaje llano (h): 57676413081093003148005107550719583540116985236696423860923466490497932824681



Para generar la firma digital $s$, Alicia debe de utilizar el mismo algoritmo que utiliza para descifrar mensajes. En otras palabras, Alicia debe calcular
$$
s = h^{d} \mod n
$$
y ya con esta información puede generar el *mensaje firmado*
$$
\text{mensaje\_firmado} = (m, s)
$$

In [11]:
s = rsa_decryption(private_key, h)
signed_message = (m, s)

print(f"El mensaje firmado es: {signed_message}\n")

TypeError: rsa_decryption() missing 1 required positional argument: 'encrypted_message'

Para verificar la firma, Beto aplica ahora el mismo algoritmo que utilizaría para encriptar mensajes. De manera más precisa, Beto calcula el Hash
$$
h = \operatorname{Hash}(m)
$$
y calcula también
$$
\tilde{h} = s^{e} \mod n
$$
Si estos dos números son iguales, entonces la firma es válida.

In [ ]:
h = sha256_of_sentence(signed_message[0])
s = signed_message[1]
h_tilde = rsa_encryption(public_key, s)

if h == h_tilde:
    verificacion_firma = "la firma es válida"
else:
    verificacion_firma = "la firma es inválida"

print(f"El valor de h es: {h}\n")
print(f"El valor de h̃ es: {h_tilde}\n")
print("Por lo tanto,", verificacion_firma)